In [1]:
from dandi.dandiapi import DandiAPIClient
from pynwb import NWBHDF5IO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pprint import pprint
from pathlib import Path
from tqdm.auto import tqdm
from itertools import islice
import requests
import textwrap


# -----------------------------
# Config
# -----------------------------

DANDISET_ID = "000006"
VERSION = "draft"

MAX_ASSETS_TO_SCAN = 20
ASSET_INDEX_TO_OPEN = 0

CACHE_DIR = Path("./dandi_nwb_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

TABLE_PREVIEW_ROWS = 5
ARRAY_PREVIEW_N = 10


# -----------------------------
# Utility printing helpers
# -----------------------------

def section(title: str):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def subsection(title: str):
    print("\n" + "-" * 90)
    print(title)
    print("-" * 90)


def short(x, n=500):
    s = str(x)
    return s if len(s) <= n else s[:n] + " ..."


def safe_getattr(obj, name, default=None):
    try:
        return getattr(obj, name)
    except Exception:
        return default


def describe_data_like(data, preview_n=ARRAY_PREVIEW_N):
    """
    Print shape/dtype for array-like objects without loading the full thing.
    """
    if data is None:
        print("  data: None")
        return

    shape = safe_getattr(data, "shape", None)
    dtype = safe_getattr(data, "dtype", None)

    print(f"  data object type: {type(data)}")
    print(f"  shape: {shape}")
    print(f"  dtype: {dtype}")

    try:
        if shape is not None and len(shape) > 0:
            sl = tuple(slice(0, min(preview_n, s)) for s in shape[:1])
            preview = np.asarray(data[sl])
        else:
            preview = np.asarray(data)
        print(f"  preview: {preview[:preview_n] if preview.ndim > 0 else preview}")
    except Exception as e:
        print(f"  preview: <could not preview safely: {type(e).__name__}: {e}>")


def preview_dynamic_table(table, name, rows=TABLE_PREVIEW_ROWS):
    """
    Preview NWB DynamicTable-like objects such as units, trials, electrodes.
    """
    subsection(f"DynamicTable: {name}")

    if table is None:
        print("  None")
        return

    print(f"  type: {type(table)}")

    try:
        print(f"  number of rows: {len(table)}")
    except Exception:
        pass

    try:
        print(f"  columns: {list(table.colnames)}")
    except Exception:
        pass

    try:
        df = table.to_dataframe()
        print(f"  dataframe shape: {df.shape}")
        print(df.head(rows))
    except Exception as e:
        print(f"  could not convert to dataframe: {type(e).__name__}: {e}")


def preview_timeseries(ts, name):
    """
    Preview NWB TimeSeries-like objects.
    """
    subsection(f"TimeSeries-like object: {name}")

    print(f"  type: {type(ts)}")
    print(f"  name: {safe_getattr(ts, 'name')}")
    print(f"  description: {short(safe_getattr(ts, 'description'))}")
    print(f"  comments: {short(safe_getattr(ts, 'comments'))}")
    print(f"  unit: {safe_getattr(ts, 'unit')}")
    print(f"  conversion: {safe_getattr(ts, 'conversion')}")
    print(f"  resolution: {safe_getattr(ts, 'resolution')}")

    rate = safe_getattr(ts, "rate")
    starting_time = safe_getattr(ts, "starting_time")
    timestamps = safe_getattr(ts, "timestamps")

    if rate is not None:
        print(f"  rate: {rate}")
        print(f"  starting_time: {starting_time}")
    elif timestamps is not None:
        print("  timestamps:")
        describe_data_like(timestamps)
    else:
        print("  timing: <no rate or timestamps found>")

    data = safe_getattr(ts, "data")
    print("  data:")
    describe_data_like(data)


def walk_container(container, name, max_depth=2, depth=0):
    """
    Recursively inspect NWB containers, but keep it shallow.
    """
    indent = "  " * depth

    if depth > max_depth:
        print(f"{indent}{name}: <max depth reached>")
        return

    print(f"{indent}{name}: {type(container)}")

    # If it looks like a TimeSeries, preview it specially.
    if hasattr(container, "data") and (hasattr(container, "timestamps") or hasattr(container, "rate")):
        preview_timeseries(container, name)
        return

    # If it looks like a DynamicTable, preview it specially.
    if hasattr(container, "to_dataframe") and hasattr(container, "colnames"):
        preview_dynamic_table(container, name)
        return

    # If it has child containers in .fields, show those.
    fields = safe_getattr(container, "fields", {})
    if isinstance(fields, dict) and fields:
        for k, v in fields.items():
            if k in {"data", "timestamps"}:
                continue
            print(f"{indent}  field: {k} -> {type(v)}")


def print_mapping(mapping, title, max_items=50):
    section(title)

    if mapping is None:
        print("None")
        return

    try:
        keys = list(mapping.keys())
    except Exception:
        print(f"Not a mapping: {type(mapping)}")
        return

    if not keys:
        print("<empty>")
        return

    print(f"Found {len(keys)} item(s):")
    for i, key in enumerate(keys[:max_items]):
        obj = mapping[key]
        print(f"\n[{i}] {key}")
        print(f"  type: {type(obj)}")
        print(f"  neurodata_type: {safe_getattr(obj, 'neurodata_type')}")
        print(f"  description: {short(safe_getattr(obj, 'description'))}")

        if hasattr(obj, "data") and (hasattr(obj, "timestamps") or hasattr(obj, "rate")):
            print(f"  shape: {safe_getattr(safe_getattr(obj, 'data'), 'shape')}")
            print(f"  dtype: {safe_getattr(safe_getattr(obj, 'data'), 'dtype')}")

    if len(keys) > max_items:
        print(f"\n... truncated after {max_items} items")


# -----------------------------
# DANDI helpers
# -----------------------------

def get_asset_size(asset):
    for attr in ["size", "blob", "metadata"]:
        try:
            value = getattr(asset, attr)
            if attr == "size" and value is not None:
                return value
        except Exception:
            pass

    try:
        meta = asset.get_raw_metadata()
        return meta.get("contentSize")
    except Exception:
        return None


def get_content_url(asset):
    """
    Get downloadable URL for a DANDI asset.
    The DANDI API has changed slightly across versions, so this is defensive.
    """
    try:
        return asset.get_content_url(follow_redirects=1, strip_query=False)
    except TypeError:
        try:
            return asset.get_content_url(follow_redirects=True)
        except TypeError:
            return asset.get_content_url()


def download_asset(asset, out_path: Path):
    """
    Download one asset to local cache.
    Uses asset.download if available; otherwise falls back to requests.
    """
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"Using cached file: {out_path}")
        return out_path

    print(f"Downloading to: {out_path}")

    try:
        asset.download(str(out_path))
        return out_path
    except Exception as e:
        print(f"asset.download failed; falling back to HTTP download.")
        print(f"Reason: {type(e).__name__}: {e}")

    url = get_content_url(asset)

    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))

        with open(out_path, "wb") as f:
            with tqdm(total=total, unit="B", unit_scale=True, desc="download") as pbar:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))

    return out_path


def safe_asset_filename(asset):
    path = getattr(asset, "path", "asset.nwb")
    return path.replace("/", "__")


# -----------------------------
# NWB inspection
# -----------------------------

def inspect_nwb(nwbfile):
    section("NWB FILE SUMMARY")

    print(f"session_description: {nwbfile.session_description}")
    print(f"identifier: {nwbfile.identifier}")
    print(f"session_start_time: {nwbfile.session_start_time}")
    print(f"timestamps_reference_time: {nwbfile.timestamps_reference_time}")
    print(f"institution: {nwbfile.institution}")
    print(f"lab: {nwbfile.lab}")
    print(f"experimenter: {nwbfile.experimenter}")
    print(f"experiment_description: {short(nwbfile.experiment_description)}")
    print(f"related_publications: {short(nwbfile.related_publications)}")
    print(f"keywords: {nwbfile.keywords}")

    section("SUBJECT")
    if nwbfile.subject is None:
        print("No subject metadata.")
    else:
        subject = nwbfile.subject
        for attr in [
            "subject_id",
            "species",
            "genotype",
            "sex",
            "age",
            "date_of_birth",
            "description",
            "strain",
        ]:
            print(f"{attr}: {safe_getattr(subject, attr)}")

    section("TOP-LEVEL NWB OBJECTS")

    top_level = {
        "acquisition": nwbfile.acquisition,
        "stimulus": nwbfile.stimulus,
        "stimulus_template": nwbfile.stimulus_template,
        "processing": nwbfile.processing,
        "analysis": nwbfile.analysis,
        "intervals": nwbfile.intervals,
        "devices": nwbfile.devices,
        "electrode_groups": nwbfile.electrode_groups,
        "imaging_planes": nwbfile.imaging_planes,
        "ogen_sites": nwbfile.ogen_sites,
    }

    for name, obj in top_level.items():
        try:
            keys = list(obj.keys())
            print(f"{name}: {len(keys)} item(s)")
            for key in keys[:20]:
                print(f"  - {key}: {type(obj[key])}")
            if len(keys) > 20:
                print("  ...")
        except Exception:
            print(f"{name}: {type(obj)}")

    print_mapping(nwbfile.acquisition, "ACQUISITION")
    print_mapping(nwbfile.stimulus, "STIMULUS")
    print_mapping(nwbfile.stimulus_template, "STIMULUS TEMPLATE")
    print_mapping(nwbfile.processing, "PROCESSING MODULES")
    print_mapping(nwbfile.intervals, "INTERVALS")

    section("CORE TABLES")

    preview_dynamic_table(safe_getattr(nwbfile, "trials"), "trials")
    preview_dynamic_table(safe_getattr(nwbfile, "units"), "units")
    preview_dynamic_table(safe_getattr(nwbfile, "electrodes"), "electrodes")
    preview_dynamic_table(safe_getattr(nwbfile, "epochs"), "epochs")
    preview_dynamic_table(safe_getattr(nwbfile, "invalid_times"), "invalid_times")

    section("DETAILED ACQUISITION PREVIEW")
    for name, obj in list(nwbfile.acquisition.items())[:10]:
        walk_container(obj, f"acquisition/{name}", max_depth=2)

    section("DETAILED PROCESSING PREVIEW")
    for module_name, module in list(nwbfile.processing.items())[:10]:
        subsection(f"processing/{module_name}")
        print(f"description: {safe_getattr(module, 'description')}")
        try:
            for obj_name, obj in list(module.data_interfaces.items())[:10]:
                walk_container(obj, f"processing/{module_name}/{obj_name}", max_depth=2)
        except Exception as e:
            print(f"Could not inspect processing module: {type(e).__name__}: {e}")


def make_mousehash_role_guess(nwbfile):
    """
    Tiny first-pass role detector.
    This is intentionally heuristic.
    The point is to map NWB contents to MouseHash-style roles.
    """
    section("MOUSEHASH ROLE GUESS")

    roles = {
        "metadata": [],
        "neural_data": [],
        "stimuli": [],
        "behavior": [],
        "conditions": [],
        "time_organization": [],
    }

    # Metadata
    if nwbfile.subject is not None:
        roles["metadata"].append("subject")
    if len(nwbfile.devices) > 0:
        roles["metadata"].append("devices")
    if len(nwbfile.electrode_groups) > 0:
        roles["metadata"].append("electrode_groups")
    if safe_getattr(nwbfile, "electrodes") is not None:
        roles["metadata"].append("electrodes table")

    # Neural data
    if safe_getattr(nwbfile, "units") is not None:
        try:
            if len(nwbfile.units) > 0:
                roles["neural_data"].append("units/spike times")
        except Exception:
            roles["neural_data"].append("units table")

    for name, obj in nwbfile.acquisition.items():
        neurodata_type = str(safe_getattr(obj, "neurodata_type", ""))
        lname = name.lower()
        if any(s in neurodata_type.lower() for s in ["electricalseries", "spike", "twophoton", "onephoton", "image"]):
            roles["neural_data"].append(f"acquisition/{name} ({neurodata_type})")
        elif any(s in lname for s in ["behavior", "position", "pupil", "eye", "running", "lick"]):
            roles["behavior"].append(f"acquisition/{name}")
        else:
            roles["neural_data"].append(f"acquisition/{name} ({neurodata_type})")

    # Stimuli
    for name in nwbfile.stimulus.keys():
        roles["stimuli"].append(f"stimulus/{name}")
    for name in nwbfile.stimulus_template.keys():
        roles["stimuli"].append(f"stimulus_template/{name}")

    # Conditions and time organization
    if safe_getattr(nwbfile, "trials") is not None:
        roles["conditions"].append("trials table")
        roles["time_organization"].append("trials/start_time, stop_time")

    for name in nwbfile.intervals.keys():
        roles["conditions"].append(f"intervals/{name}")
        roles["time_organization"].append(f"intervals/{name}")

    for name, obj in nwbfile.acquisition.items():
        if hasattr(obj, "timestamps") or hasattr(obj, "rate"):
            roles["time_organization"].append(f"timing for acquisition/{name}")

    pprint(roles)


# -----------------------------
# Main script
# -----------------------------

client = DandiAPIClient.for_dandi_instance("dandi")
dandiset = client.get_dandiset(DANDISET_ID, VERSION)

print("Dandiset:", dandiset.identifier)
print("Version:", VERSION)

# Dandiset metadata
try:
    dandiset_meta = dandiset.get_raw_metadata()
except Exception:
    dandiset_meta = dandiset.get_metadata()

section("DANDISET METADATA")
print("Dandiset metadata type:", type(dandiset_meta))
pprint(dandiset_meta if isinstance(dandiset_meta, dict) else str(dandiset_meta)[:2000])


section("SCANNING ASSETS")

assets = list(islice(dandiset.get_assets(), MAX_ASSETS_TO_SCAN))
nwb_assets = [a for a in assets if getattr(a, "path", "").endswith(".nwb")]

print(f"Scanned {len(assets)} asset(s).")
print(f"Found {len(nwb_assets)} NWB asset(s) among first {MAX_ASSETS_TO_SCAN} asset(s).")

for i, asset in enumerate(nwb_assets):
    size = get_asset_size(asset)
    print(f"\n[{i}] {asset.path}")
    print(f"    asset_id: {asset.identifier}")
    print(f"    size: {size}")

if not nwb_assets:
    raise RuntimeError(
        "No NWB files found in the scanned assets. "
        "Increase MAX_ASSETS_TO_SCAN or check the dandiset/version."
    )

if ASSET_INDEX_TO_OPEN >= len(nwb_assets):
    raise IndexError(
        f"ASSET_INDEX_TO_OPEN={ASSET_INDEX_TO_OPEN}, "
        f"but only found {len(nwb_assets)} NWB asset(s)."
    )

asset = nwb_assets[ASSET_INDEX_TO_OPEN]

section("SELECTED ASSET")
print("path:", asset.path)
print("asset_id:", asset.identifier)
print("size:", get_asset_size(asset))

local_path = CACHE_DIR / safe_asset_filename(asset)
download_asset(asset, local_path)

section("OPENING NWB FILE")
print("local_path:", local_path)
print("file_size_MB:", local_path.stat().st_size / 1e6)

with NWBHDF5IO(str(local_path), mode="r", load_namespaces=True) as io:
    nwbfile = io.read()
    inspect_nwb(nwbfile)
    make_mousehash_role_guess(nwbfile)

print("\nDone.")

/home/maria/mousehash/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dandiset: 000006
Version: draft

DANDISET METADATA
Dandiset metadata type: <class 'dict'>
{'@context': 'https://raw.githubusercontent.com/dandi/schema/master/releases/0.6.0/context.json',
 'about': [{'identifier': 'http://purl.obolibrary.org/obo/UBERON_0016634',
            'name': 'premotor cortex',
            'schemaKey': 'Anatomy'},
           {'identifier': 'https://purl.brain-bican.org/ontology/dhbao/DHBA_12773',
            'name': 'pyramidal tract',
            'schemaKey': 'Anatomy'}],
 'access': [{'contactPoint': {'schemaKey': 'ContactPoint'},
             'schemaKey': 'AccessRequirements',
             'status': 'dandi:OpenAccess'}],
 'assetsSummary': {'approach': [{'name': 'electrophysiological approach',
                                 'schemaKey': 'ApproachType'},
                                {'name': 'behavioral approach',
                                 'schemaKey': 'ApproachType'}],
                   'dataStandard': [{'identifier': 'RRID:SCR_015242',
            